# AIC25 — Single-Camera Tracking Consistency (cross-scene)

Trains a **learned ReID tracklet matcher** on one warehouse and tests it on a **different, unseen** one — the rigorous generalisation test. Compares **raw → conservative (heuristic) → learned** against ground truth (IDF1, ID switches, fragmentations).

**How to use:** set `TRAIN_SCENE` / `TEST_SCENE` in the Configuration cell, then **Run all**. Everything (single-camera data, embeddings, matcher, results) is **cached to Drive**, so re-runs skip finished work.

Branch: **main**. T4 GPU (only the data-generation step needs it; benchmarking is CPU).

---
## Step 0 — Environment + Drive

In [ ]:
import os, sys, shutil
ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if ON_COLAB:
    REPO='/content/repo'; PY='python'; DRIVE='/content/drive/MyDrive/AIC25'
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'): drive.mount('/content/drive')
    else: print('Drive already mounted.')
    for d in ['models','outputs/Detection','outputs/Tracking']:
        os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
    print('Colab | Drive:', DRIVE)
else:
    REPO='/home/seco/deepLearning/Single-Camera-Tracking-Consistency'; PY=f'{REPO}/.venv/bin/python'; DRIVE=None
    os.chdir(REPO); print('Local')
print('REPO:', REPO)

---
## Step 1 — Clone + checkout repo branch + install

In [ ]:
if ON_COLAB:
    import subprocess as _sp
    BRANCH = 'main'
    GIT_URL = 'https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git'

    def _run(cmd, check=True):
        result = _sp.run(cmd, capture_output=True, text=True)
        if check and result.returncode != 0:
            detail = (result.stderr or result.stdout).strip()
            raise RuntimeError(f"{' '.join(cmd)} failed:\n{detail}")
        return result

    if not os.path.exists(f'{REPO}/.git'):
        if os.path.exists(REPO): shutil.rmtree(REPO)
        _run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GIT_URL, REPO])
    else:
        _run(['git', '-C', REPO, 'fetch', 'origin', BRANCH])
        _run(['git', '-C', REPO, 'clean', '-fd', 'tracklet_repair/models', 'tracklet_repair/results'], check=False)
        _run(['git', '-C', REPO, 'checkout', '-f', BRANCH], check=False)
        _run(['git', '-C', REPO, 'reset', '--hard', f'origin/{BRANCH}'], check=False)

    os.chdir(REPO)
    if not os.path.isdir(f'{REPO}/tracklet_repair'):
        raise RuntimeError('tracklet_repair missing after checkout of main')
    print('On main ✓')
    setup_marker = f'{REPO}/.colab_deps_ok_tier2'
    if os.path.exists(setup_marker):
        print('Dependencies already installed for this runtime.')
    else:
        for _p in ['thop','loguru','lap','motmetrics','filterpy','easydict','yacs','termcolor',
                   'prettytable','tabulate','ninja','cython_bbox','pycocotools','huggingface_hub','h5py']:
            _sp.run(['pip','install','-q',_p], capture_output=True, text=True)
        if _sp.run(['pip','install','-q','faiss-gpu'], capture_output=True).returncode != 0:
            _sp.run(['pip','install','-q','faiss-cpu'], capture_output=True)
        os.chdir(f'{REPO}/BoT-SORT');         os.system('python setup.py develop --quiet 2>/dev/null')
        os.chdir(f'{REPO}/deep-person-reid'); os.system('python setup.py develop --quiet 2>/dev/null')
        os.chdir(REPO); os.system('pip install -q -r tracking/requirements.txt 2>/dev/null')
        open(setup_marker, 'w').write('ok\n')
    print('Dependencies installed.')
else:
    print('Local: skip.')

---
## Step 2 — GPU check

In [ ]:
import subprocess
r = subprocess.run([PY,'-c','import torch; print("CUDA:", torch.cuda.is_available())'], capture_output=True, text=True)
print(r.stdout.strip())
if ON_COLAB and 'CUDA: False' in r.stdout:
    raise RuntimeError('NO GPU — switch to T4, Restart, re-run.')

---
## Step 3 — Models (OSNet from HF mirror + ByteTrack; AIC25 detector if trained)

In [ ]:
if ON_COLAB:
    from huggingface_hub import hf_hub_download
    M=f'{DRIVE}/models'; os.makedirs(M, exist_ok=True)
    osnet_local=f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar'; osnet_drive=f'{M}/osnet_ms_m_c.pth.tar'
    os.makedirs(os.path.dirname(osnet_local), exist_ok=True)
    if os.path.exists(osnet_local): print('OSNet: local')
    elif os.path.exists(osnet_drive): shutil.copy(osnet_drive, osnet_local); print('OSNet: from Drive')
    else:
        fn='osnet_x1_0_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth'
        src=hf_hub_download(repo_id='kaiyangzhou/osnet', filename=fn)
        shutil.copy(src, osnet_local); shutil.copy(osnet_local, osnet_drive); print('OSNet: from HF mirror')
    bt_local=f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'; bt_drive=f'{M}/bytetrack_x_mot17.pth.tar'
    os.makedirs(os.path.dirname(bt_local), exist_ok=True)
    if not os.path.exists(bt_local):
        if os.path.exists(bt_drive): shutil.copy(bt_drive, bt_local)
        else:
            os.system('pip install -q -U gdown'); import gdown
            gdown.download(id='1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5', output=bt_local, quiet=False)
            if os.path.exists(bt_local): shutil.copy(bt_local, bt_drive)
    aic=f'{M}/ai_city_ckpt.pth.tar'
    if os.path.exists(aic): shutil.copy(aic, f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'); print('AIC25 detector: from Drive ✓')
    else: print('AIC25 detector: not trained — ByteTrack fallback (HOTA will be lower)')
else: print('Local: models in place.')

---
## Configuration
Set the train/test scenes here — everything below uses these globals. Never set them equal (no train+test on the same scene).

In [ ]:
# ===== CONFIGURATION — edit ONLY here =====
DATASET      = 'Val'
TRAIN_SCENE  = 'Warehouse_016'   # the matcher LEARNS from this scene
TEST_SCENE   = 'Warehouse_015'   # the matcher is TESTED on this (unseen) scene
CAMERAS      = ['Camera','Camera_01','Camera_02','Camera_03']   # None = all 12
MAXF         = 1500              # frames per camera (0 = all 9000)
TRACK_PARAMS = {}                # BoT-SORT overrides, e.g. {'match_thresh':0.9,'track_buffer':60}
os.chdir(REPO)
assert TRAIN_SCENE != TEST_SCENE, 'TRAIN_SCENE and TEST_SCENE must differ (never train+test the same scene).'
print(f'TRAIN on {TRAIN_SCENE}  ->  TEST on {TEST_SCENE} | cams {CAMERAS or "ALL"} | cap {MAXF or "ALL"}')

---
## Build the runner  *(all logic in tracklet_repair/src/pipeline/cross_scene_runner.py)*

In [ ]:
import getpass
HF_TOKEN=''
try:
    from google.colab import userdata; HF_TOKEN=userdata.get('HF_TOKEN') or ''
except Exception: pass
if ON_COLAB and not HF_TOKEN: HF_TOKEN=getpass.getpass('HF token: ')
from tracklet_repair.src.pipeline.cross_scene_runner import CrossSceneRunner
runner = CrossSceneRunner(repo=REPO, py=PY, dataset=DATASET, cameras=CAMERAS, maxf=MAXF,
                          track_params=TRACK_PARAMS, hf_token=HF_TOKEN, on_colab=ON_COLAB, drive=DRIVE)
print('runner ready')

---
## Step 1 — Link Drive + download TRAIN scene

In [ ]:
runner.link_outputs()
runner.download_scene(TRAIN_SCENE)

## Step 2 — Single-camera tracking, TRAIN scene  *(GPU; long; cached to Drive)*

In [ ]:
runner.generate_scene(TRAIN_SCENE)

## Step 3 — Train the matcher  *(CPU; saved to Drive)*

In [ ]:
runner.train_on(TRAIN_SCENE)

## Step 4 — Download TEST scene

In [ ]:
runner.download_scene(TEST_SCENE)

## Step 5 — Single-camera tracking, TEST scene  *(GPU; long; cached to Drive)*

In [ ]:
runner.generate_scene(TEST_SCENE)

## Step 6 — Test + save results  *(CPU; TRAIN matcher on unseen TEST scene)*

In [ ]:
runner.benchmark(TEST_SCENE, TRAIN_SCENE)